# Component 2 — NEW Vision trainer CPU verification (v2)

**Prepared, not executed in Colab.** This is a new source snapshot, separate from the earlier 8 model/Grad-CAM passes.

Use a fresh standard CPU runtime (Hardware accelerator: None). Run the four code cells in order. Cell 2 asks for `vision_training_colab_bundle_v2.zip` only. No datasets, Drive mount, Git clone, package installation or pretrained downloads.

First execute the seven formerly skipped trainer backend cases, then the remaining 44 focused approval/leakage/metric checks. Acceptance: **51 passed, 0 failed, 0 errors, 0 skipped**, with matching source hashes before and after testing. A failure/skip stops further jobs and preserves actual evidence.

Architecture instances use `pretrained=False` (`weights=None`). Tiny generated-image optimisation and checkpoint tests verify plumbing, not dental accuracy or five-class clinical validity. All real labels, original splits, MGI 0–4 definitions and pending reviews remain unchanged.

Run the evidence-export cell even after a dependency, upload or test failure. Return the downloaded `vision_training_runtime_evidence_v2.zip` for review. Do not weaken tests or auto-reinstall packages.

In [ ]:
# Cell 1: discover installed dependencies and verify CPU compatibility; install nothing.
import importlib.util, importlib.metadata, json, os, platform, sys, tempfile, traceback
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
SESSION = Path(tempfile.mkdtemp(prefix='rp_vision_trainer_v2_', dir='/content'))
RESULTS = SESSION / 'results'
RESULTS.mkdir()
status = {'version': 'vision-training-runtime-v2', 'stage': 'dependency_check',
          'verified': False, 'jobs': {}, 'colab_tests_executed': False}
def save_status():
    (RESULTS / 'verification_summary.json').write_text(json.dumps(status, indent=2))
save_status()
required = {'torch': 'torch', 'torchvision': 'torchvision', 'PIL': 'Pillow',
            'numpy': 'numpy', 'pytest': 'pytest'}
runtime_info = {'python': platform.python_version(), 'test_device': 'cpu',
                'platform': platform.platform(), 'cpu_threads_requested': 2}
for module, distribution in required.items():
    try:
        runtime_info[distribution] = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        runtime_info[distribution] = None
try:
    missing = [name for name in required if importlib.util.find_spec(name) is None]
    if missing:
        raise RuntimeError('Missing packages: ' + ', '.join(missing) + '; stop, no auto-install.')
    import torch, torchvision, numpy as np, PIL, pytest
    from PIL import Image
    from torchvision.ops import nms
    torch.set_num_threads(2)
    runtime_info.update(torch=str(torch.__version__), torchvision=str(torchvision.__version__),
                        numpy=np.__version__, Pillow=PIL.__version__, pytest=pytest.__version__,
                        cpu_threads_actual=torch.get_num_threads())
    probe = np.arange(4, dtype=np.float32)
    np.testing.assert_array_equal(torch.from_numpy(probe).numpy(), probe)
    if nms(torch.tensor([[0., 0., 1., 1.]]), torch.tensor([1.]), 0.5).tolist() != [0]:
        raise RuntimeError('CPU torchvision compiled-op compatibility probe failed')
    if np.asarray(Image.new('RGB', (6, 4))).shape != (4, 6, 3):
        raise RuntimeError('RGB Pillow probe failed')
    runtime_info['compatibility_probes'] = 'imports, NumPy bridge, CPU torchvision op, RGB Pillow passed'
    status['stage'] = 'dependencies_ready'
except Exception:
    message = traceback.format_exc()
    (RESULTS / 'compatibility_failure.txt').write_text(message)
    status.update(stage='dependency_failure', execution_error=message)
    print(message)
    raise
finally:
    (RESULTS / 'runtime_versions.json').write_text(json.dumps(runtime_info, indent=2))
    save_status()
    print(json.dumps(runtime_info, indent=2))
    print('Evidence directory:', RESULTS)


In [ ]:
# Cell 2: upload the NEW code-only ZIP and verify every source before writing it.
from google.colab import files
import hashlib, io, zipfile
EXPECTED_ZIP = 'vision_training_colab_bundle_v2.zip'
EXPECTED_BUNDLE_SHA256 = '7b4d1c89e8f0465739b1ab4255465ccd6b8f644997f3c6a7f72a8a520b584cd0'
EXPECTED_FILES = {'src/vision/model_pipeline.py': {'sha256': 'c11ad18912575f3567d0baf0eefb1bfbb5eb00fbfa075880fe84aef1ccd34b3c', 'bytes': 12298}, 'src/vision/train_pipeline.py': {'sha256': 'e7a979b0875af0e459496bca79ce3c80e594acf2639ea7dc9d8659ec75ffc695', 'bytes': 24342}, 'tests/test_vision_training.py': {'sha256': '797f70b29dd968a2b0e6d0b1528867bf115a68467f57b806cf779de024a3dedb', 'bytes': 15303}}
try:
    if status['stage'] != 'dependencies_ready':
        raise RuntimeError('Run the successful dependency check first')
    uploaded = files.upload()
    if set(uploaded) != {EXPECTED_ZIP}:
        raise RuntimeError('Upload exactly ' + EXPECTED_ZIP + '; no dataset or older bundle')
    payload = uploaded[EXPECTED_ZIP]
    if hashlib.sha256(payload).hexdigest() != EXPECTED_BUNDLE_SHA256:
        raise RuntimeError('Bundle SHA-256 mismatch; stop')
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        names = archive.namelist()
        allowed = set(EXPECTED_FILES) | {'runtime_source_manifest.json'}
        if len(names) != len(allowed) or set(names) != allowed:
            raise RuntimeError('Unexpected or repeated ZIP entries')
        if sum(info.file_size for info in archive.infolist()) > 1_000_000:
            raise RuntimeError('Unexpected extraction size')
        source_manifest = json.loads(archive.read('runtime_source_manifest.json'))
        if (source_manifest['source_branch'] != 'vinuthan'
                or source_manifest['version'] != 'vision-training-runtime-v2'
                or source_manifest['files'] != EXPECTED_FILES):
            raise RuntimeError('Source manifest mismatch')
        verified_sources = {}
        for name, expected in EXPECTED_FILES.items():
            content = archive.read(name)
            if len(content) != expected['bytes'] or hashlib.sha256(content).hexdigest() != expected['sha256']:
                raise RuntimeError('Source SHA-256/size mismatch: ' + name)
            verified_sources[name] = content
    ROOT = SESSION / 'code'
    ROOT.mkdir(exist_ok=False)
    for name, content in verified_sources.items():
        destination = ROOT / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        with destination.open('xb') as stream:
            stream.write(content)
    (RESULTS / 'runtime_source_manifest.json').write_text(json.dumps(source_manifest, indent=2))
    (RESULTS / 'bundle_sha256.txt').write_text(EXPECTED_BUNDLE_SHA256 + '\n')
    status.update(stage='source_ready', source_hashes_verified=True)
    print('Verified NEW uncommitted trainer snapshot:', ROOT)
    print(json.dumps(source_manifest, indent=2))
except Exception:
    message = traceback.format_exc()
    (RESULTS / 'upload_failure.txt').write_text(message)
    status.update(stage='upload_failure', verified=False, execution_error=message)
    print(message)
    raise
finally:
    save_status()


In [ ]:
# Cell 3: existing focused tests only, generated RGB, CPU, no downloaded weights.
import subprocess, xml.etree.ElementTree as ET
backend_names = ['test_frozen_backbone_bn_and_validation_no_grad', 'test_selected_batchnorm_updates_only_approved_module', 'test_checkpoint_resume_matches_uninterrupted_and_reproducible', 'test_early_stopping_and_best_checkpoint_reload', 'test_actual_architecture_generated_train_validation_step', 'test_generated_smoke_supports_checkpoint_reload_and_resume']
backend_targets = ['tests/test_vision_training.py::' + name for name in backend_names]
exclude_backend = 'not (' + ' or '.join(backend_names) + ')'
jobs = [('trainer_backend', 7, backend_targets),
        ('approval_and_metrics', 44, ['tests/test_vision_training.py', '-k', exclude_backend])]
def source_hashes():
    return {name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest() for name in EXPECTED_FILES}
def count_xml(path):
    if not path.exists():
        return {'passed': None, 'failed': None, 'errors': None, 'skipped': None}
    cases = ET.parse(path).getroot().findall('.//testcase')
    failed = sum(case.find('failure') is not None for case in cases)
    errors = sum(case.find('error') is not None for case in cases)
    skipped = sum(case.find('skipped') is not None for case in cases)
    return {'passed': len(cases) - failed - errors - skipped,
            'failed': failed, 'errors': errors, 'skipped': skipped}
try:
    if status['stage'] != 'source_ready':
        raise RuntimeError('Successful dependency and source checks are required')
    environment = dict(os.environ, PYTHONPATH=str(ROOT), PYTEST_DISABLE_PLUGIN_AUTOLOAD='1',
                       PYTHONDONTWRITEBYTECODE='1', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2',
                       CUDA_VISIBLE_DEVICES='')
    before = source_hashes()
    if any(before[name] != expected['sha256'] for name, expected in EXPECTED_FILES.items()):
        raise RuntimeError('Source changed before tests')
    status.update(stage='testing', verified=False, source_hashes_before=before)
    save_status()
    for label, expected, targets in jobs:
        xml_path, log_path = RESULTS / (label + '.xml'), RESULTS / (label + '.log')
        command = [sys.executable, '-B', '-m', 'pytest', '-q', '-rs', '--color=no',
                   '-p', 'no:cacheprovider', '--junitxml', str(xml_path), *targets]
        summary = {'expected_tests': expected, 'verified': False, 'returncode': None,
                   'passed': None, 'failed': None, 'errors': None, 'skipped': None,
                   'command': command}
        print('RUN', command)
        status['colab_tests_executed'] = True
        try:
            process = subprocess.run(command, cwd=ROOT, env=environment,
                                     capture_output=True, text=True, timeout=600)
            output = process.stdout + process.stderr
            log_path.write_text(output)
            print(output)
            summary.update(returncode=process.returncode, **count_xml(xml_path))
            summary['verified'] = (process.returncode == 0 and summary['passed'] == expected
                                   and summary['failed'] == summary['errors'] == summary['skipped'] == 0)
        except Exception as error:
            message = traceback.format_exc()
            captured = ''
            if isinstance(error, subprocess.TimeoutExpired):
                for part in (error.stdout, error.stderr):
                    if part:
                        captured += part.decode(errors='replace') if isinstance(part, bytes) else part
            previous = log_path.read_text() if log_path.exists() else ''
            log_path.write_text(previous + captured + message)
            summary['execution_error'] = message
            try:
                summary.update(count_xml(xml_path))
            except Exception:
                summary['xml_parse_error'] = traceback.format_exc()
            print(message)
        status['jobs'][label] = summary
        save_status()
        if not summary['verified']:
            break  # Preserve failure/skip evidence; do not rerun or weaken checks automatically.
    after = source_hashes()
    status['source_hashes_after'] = after
    status['source_hashes_unchanged'] = after == before
    successful = (len(status['jobs']) == 2 and all(s['verified'] for s in status['jobs'].values())
                  and after == before)
    status.update(verified=successful, stage='verified' if successful else 'not_verified')
except Exception:
    message = traceback.format_exc()
    (RESULTS / 'execution_failure.txt').write_text(message)
    status.update(verified=False, stage='execution_failure', execution_error=message)
    print(message)
finally:
    save_status()
    print(json.dumps(status, indent=2))
    if status['verified']:
        print('VERIFIED: 7 trainer backend + 44 focused checks passed; zero failures/errors/skips.')
        print('Synthetic runtime plumbing only; no dental training or final manifest approval.')
    else:
        print('NOT VERIFIED. Run the evidence-export cell and retain exact logs/XML.')


In [ ]:
# Cell 4: export actual evidence even after a failure; never include checkpoint weights.
from google.colab import files
import tempfile, zipfile
if 'RESULTS' not in globals() or not RESULTS.exists():
    raise RuntimeError('No runtime evidence yet: run Cell 1 first')
export_dir = Path(tempfile.mkdtemp(prefix='export_', dir=SESSION))
evidence = export_dir / 'vision_training_runtime_evidence_v2.zip'
allowed_evidence = ['runtime_versions.json', 'runtime_source_manifest.json', 'bundle_sha256.txt',
                    'verification_summary.json', 'compatibility_failure.txt', 'upload_failure.txt',
                    'execution_failure.txt', 'trainer_backend.log', 'trainer_backend.xml',
                    'approval_and_metrics.log', 'approval_and_metrics.xml']
with zipfile.ZipFile(evidence, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in allowed_evidence:
        path = RESULTS / name
        if path.is_file():
            archive.write(path, arcname=name)
print('Evidence only; keep failures/skips and source hashes:', evidence)
files.download(str(evidence))
